### 신경망 구성요소
- 입력 : 입력
- 가중치(weight) : 입력값의 중요도
- 편향(bias) : 각 뉴런이 가지는 추가적인 상수항
- 활성화 함수() : 신경망의 각 층 출력에 비선형성(non-linearity)을 추가해주는 함수
- 출력 : 결과

#### 신경망 종류
| 종류 | 특징 | 주 사용처 |
|---|---|---|
| **MLP / FCN** (다층 퍼셉트론) | 모든 노드가 다음 층 모든 노드와 연결 (Fully Connected) | 기본 구조, 표(정형) 데이터, 다른 구조의 마지막 출력층 |
| **CNN** (합성곱 신경망) | 필터(kernel)로 이미지의 지역적 패턴(엣지, 질감 등) 추출 | 이미지 분류, 객체 탐지, 이미지 처리 전반 |
| **RNN / LSTM / GRU** | 이전 시점의 정보를 기억하며 순차적으로 처리 | 시계열, 텍스트(예전 방식), 음성 |
| **Transformer** | Self-Attention으로 전체 시퀀스를 한 번에 병렬 처리, "어디에 집중할지" 학습 | 현재 NLP/LLM 표준 (GPT, BERT), 최근엔 이미지(ViT)에도 |
| **GAN** (생성적 적대 신경망) | Generator(생성자) vs Discriminator(판별자)가 서로 경쟁하며 학습 | 이미지 생성, 딥페이크, 스타일 변환 |
| **Autoencoder** | 입력을 압축(encode)했다가 복원(decode), 병목 구간에서 특징 추출 | 차원 축소, 이상 탐지, 노이즈 제거 |
| **GNN** (그래프 신경망) | 노드-엣지로 이루어진 그래프 구조 데이터 처리 | 소셜 네트워크, 분자구조, 추천시스템 |
| **Diffusion Model** | 노이즈를 점진적으로 제거해가며 이미지 생성 | 최신 이미지 생성 (Stable Diffusion, DALL-E 계열) |

In [2]:
import torch
import torch.nn as nn

In [3]:
# And 게이트 구현
def perceptron_and(x1, x2):
    w = torch.tensor([0.5, 0.5]) # 가중치
    b = torch.tensor(-0.7) # 편향
    x = torch.tensor([x1, x2], dtype=torch.float32)

    # 1차원 벡터끼리의 내적 계산 torch.dot(1D,1D)
    output = torch.dot(w, x) + b
    print(output)
    return 1 if output >= 0 else 0


print(perceptron_and(0,0))
print(perceptron_and(0,1))
print(perceptron_and(1,0))
print(perceptron_and(1,1))

tensor(-0.7000)
0
tensor(-0.2000)
0
tensor(-0.2000)
0
tensor(0.3000)
1


In [4]:
# (1*3) + (2*4) 
torch.dot(torch.tensor([1,2]), torch.tensor([3,4])) 

tensor(11)

In [5]:
# or 게이트 구현 해보기
def perceptron_or(x1,x2):
    w = torch.tensor([0.5,0.5])
    b = torch.tensor(0)
    x = torch.tensor([x1, x2], dtype=torch.float32)
    output = torch.dot(w, x) + b
    print(output)
    return 1 if output > 0 else 0

print(perceptron_or(0,0))
print(perceptron_or(0,1))
print(perceptron_or(1,0))
print(perceptron_or(1,1))

tensor(0.)
0
tensor(0.5000)
1
tensor(0.5000)
1
tensor(1.)
1


In [6]:
# xor?
def perceptron_xor(x1,x2):
    w = torch.tensor([0.5,0.5])
    b = torch.tensor(-0.5)
    x = torch.tensor([x1, x2], dtype=torch.float32)
    output = torch.dot(w, x) + b
    print(output)
    return 1 if output == 0 else 0

print(perceptron_xor(0,0))
print(perceptron_xor(0,1))
print(perceptron_xor(1,0))
print(perceptron_xor(1,1))

tensor(-0.5000)
0
tensor(0.)
1
tensor(0.)
1
tensor(0.5000)
0


##### 퍼셉트론 & XOR 정리
1. 단일 퍼셉트론으로 되는 것
- AND, OR, NAND → 직선 하나로 나눌 수 있는(선형분리 가능) 문제라서 가중치·편향 조절만으로 해결 가능

2. 단일 퍼셉트론으로 안 되는 것
- XOR → 선형분리 불가능한 문제. 좌표에 찍으면 대각선으로 꼬여있어서 직선 하나로 절대 못 나눔
- 위 xor output == 0으로 된 것처럼 보였던 건, 우연히 부동소수점이 딱 0.0으로 맞아떨어진 편법 / 진짜 해결이 아님

3. 진짜 해결책 — 은닉층 추가
- XOR = AND(NAND(x1,x2), OR(x1,x2)) 처럼 퍼셉트론을 2단으로 쌓으면 해결됨

4. 실전 방식 — 딥러닝
- 가중치를 사람이 안 정하고, nn.Module + nn.Linear로 구조만 짜고
- loss.backward() + optimizer.step()으로 모델이 스스로 가중치를 찾아냄 (역전파/경사하강법)
- 은닉 노드 수·초기화·학습률에 따라 학습이 잘 안 될 수도 있음 (로컬 미니멈)

### 활성화 함수 비교
| | Sigmoid | ReLU | Tanh |
|---|---|---|---|
| **수식** | `1 / (1 + e^-x)` | `max(0, x)` | `(e^x - e^-x) / (e^x + e^-x)` |
| **출력 범위** | 0 ~ 1 | 0 ~ ∞ | -1 ~ 1 |
| **그래프 모양** | S자, 완만하게 0→1 | 0 이하는 0, 0 이상은 직선 | S자, 0 중심 대칭 |
| **0 중심(zero-centered)?** | X | X | O |

### Sigmoid
- 출력을 0~1로 눌러줘서 확률처럼 해석 가능 → 이진 분류 출력층에 사용
- 단점: 극단값에서 기울기 소실(vanishing gradient) 발생

### ReLU
- 0 이하는 0, 0 이상은 그대로 통과 → 계산 단순, 빠름
- 기울기 소실 문제 적어서 은닉층 기본 선택
- 단점: dying ReLU (음수 입력 지속 시 뉴런이 죽음)

### Tanh
- Sigmoid를 -1~1로 확장, 0 중심이라 Sigmoid보다 학습 유리
- 극단값에서 기울기 소실은 여전히 존재
- 주로 RNN 계열에서 사용

### 순전파 (Forward Propagation)
- 입력 데이터가 신경망을 앞에서 뒤로(입력층 → 은닉층 → 출력층) 통과하면서 예측값을 계산하는 과정

In [7]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2,3)
        self.fc2 = nn.Linear(3,1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x
model = SimpleNN()
sample_input = torch.tensor([1.0, 2.0])
output = model(sample_input)
print(output)

tensor([0.4528], grad_fn=<SigmoidBackward0>)


### 순전파 vs 역전파 비교
| | 순전파 (Forward) | 역전파 (Backward) |
|---|---|---|
| 방향 | 입력 → 출력 | 출력(오차) → 입력 |
| 목적 | 예측값 계산 | 가중치 업데이트를 위한 gradient 계산 |
| 코드 | `pred = model(X)` | `loss.backward()` |
| 언제 실행 | 항상 (학습/추론 둘 다) | 학습(training) 할 때만 |